In [ ]:
import pandas as pd

orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

print(orders.shape)
print(customers.shape)

(99441, 8)
(99441, 5)


In [ ]:
# Keep only delivered orders
orders = orders[orders['order_status'] == 'delivered'].copy()

# Convert date columns to real dates
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

# Calculate delay in days
orders['delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# Join customer state onto each order
merged = orders.merge(customers[['customer_id','customer_state']], on='customer_id', how='left')

print(merged[['order_id','customer_state','delay_days']].head())

                           order_id customer_state  delay_days
0  e481f51cbdc54678b7cc49136f2d6af7             SP        -8.0
1  53cdb2fc8bc7dce0b6741e2150273451             BA        -6.0
2  47770eb9100c2d0c44946d9cf07ec65d             GO       -18.0
3  949d5b44dbf5de918fe9c16f97b45f8a             RN       -13.0
4  ad21c59c0840e6cb83a9ceb5573f8159             SP       -10.0


In [ ]:
state_summary = merged.groupby('customer_state')['delay_days'].agg(
    avg_delay='mean',
    total_orders='count',
    late_orders=lambda x: (x > 0).sum()
).reset_index()

state_summary['late_rate_percent'] = round((state_summary['late_orders'] / state_summary['total_orders']) * 100, 2)

state_summary = state_summary.sort_values('late_rate_percent', ascending=False)

print(state_summary.head(10))

   customer_state  avg_delay  total_orders  late_orders  late_rate_percent
1              AL  -8.707809           397           85              21.41
9              MA  -9.571827           717          125              17.43
24             SE -10.020896           335           51              15.22
16             PI -11.306723           476           66              13.87
5              CE -10.804535          1279          176              13.76
21             RR -17.292683            41            5              12.20
4              BA -10.794533          3256          396              12.16
18             RJ -11.761215         12350         1495              12.11
13             PA -14.066596           946          106              11.21
7              ES -10.496241          1995          214              10.73


In [ ]:
items = pd.read_csv("olist_order_items_dataset.csv")

# Add up all item prices per order, since one order can have multiple items
order_value = items.groupby('order_id')['price'].sum().reset_index()
order_value.columns = ['order_id', 'order_value']

# Attach order value to our merged delivery data
merged = merged.merge(order_value, on='order_id', how='left')

print(merged[['order_id','delay_days','order_value']].head())

                           order_id  delay_days  order_value
0  e481f51cbdc54678b7cc49136f2d6af7        -8.0        29.99
1  53cdb2fc8bc7dce0b6741e2150273451        -6.0       118.70
2  47770eb9100c2d0c44946d9cf07ec65d       -18.0       159.90
3  949d5b44dbf5de918fe9c16f97b45f8a       -13.0        45.00
4  ad21c59c0840e6cb83a9ceb5573f8159       -10.0        19.90


In [ ]:
# Only look at orders that were actually late
late_orders_df = merged[merged['delay_days'] > 0].copy()

# Apply our assumption: 10% of a late order's value is estimated "at risk"
late_orders_df['estimated_cost_at_risk'] = late_orders_df['order_value'] * 0.10

# Total estimated cost across all late orders
total_cost_at_risk = late_orders_df['estimated_cost_at_risk'].sum()

print("Number of late orders:", len(late_orders_df))
print("Total estimated cost at risk: $", round(total_cost_at_risk, 2))

Number of late orders: 6534
Total estimated cost at risk: $ 98592.43


In [ ]:
!pip install -q google-genai

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import google.generativeai as genai
genai.configure(api_key="YOUR_API_KEY_HERE")

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
!pip install -q groq

from groq import Groq

client = Groq(api_key="YOUR_API_KEY_HERE")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working."}
    ]
)

print(response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.5 MB/s eta 0:00:00
Hello. I'm working and ready to assist you with any questions or tasks you may have. How can I help you today?


In [ ]:
top_states = state_summary.head(3)

prompt = f"""
You are a logistics operations analyst reviewing Brazilian e-commerce delivery data.
The state codes below are Brazilian state abbreviations (e.g., AL = Alagoas, MA = Maranhão,
SE = Sergipe) — NOT US states. Write a short, plain-language daily brief based on this
delivery delay data. Mention the top problem states by their full Brazilian state names,
their late rate, and one clear recommendation.

Data:
{top_states.to_string(index=False)}

Total late orders: {len(late_orders_df)}
Estimated cost at risk: ${round(total_cost_at_risk, 2)}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Daily Brief: E-commerce Delivery Delays in Brazil

Our analysis of e-commerce delivery data reveals that some Brazilian states are experiencing higher rates of delayed deliveries. The top three problem states are Alagoas, Maranhão, and Sergipe, with late rates of 21.41%, 17.43%, and 15.22%, respectively.

To mitigate these delays and reduce the estimated cost at risk of $98,592.43, we recommend that logistics operations be optimized in these states. Specifically, we suggest increasing the number of delivery personnel or vehicles in Alagoas, which has the highest late rate, to ensure timely delivery of orders and improve customer satisfaction. By addressing these delays, we can reduce costs, enhance customer experience, and improve our overall logistics efficiency.


In [ ]:
# Export the state-level summary (for the delay heatmap and scorecards)
state_summary.to_csv("state_summary.csv", index=False)

# Export the full late orders detail (for deeper drill-down if needed)
late_orders_df.to_csv("late_orders_detail.csv", index=False)

print("Files created successfully")

Files created successfully
